# 10 - run pipeline stages

Generic runner for the stage scripts in `scripts/` (all of them exist). The order of the first real run, with the
`STAGE` or `STEPS` value of every step, is in `docs/RUNBOOK.md`.

What one run of this notebook does:

1. It checks that the workspace repo is a PRIVATE Hugging Face dataset repo (it stops if not), installs the `msd`
   wheel from it, and pulls `scripts/`, `pipelines/`, `configs/`, `data/` and every earlier result (`results/`).
2. If `A1_REPO` is set, it downloads A1 (the private model repo that `finetune.py --push-to` wrote) into the folder
   `A1/`, because `run_resta.py --drifted` needs a local folder.
3. It runs `python scripts/<STAGE>.py` with `STAGE_ARGS`. When `STEPS` is not empty it runs each `(script, args)` pair
   of `STEPS` instead, in order, in this one session, so a model one step makes (an A2 from `run_resta`) is still on
   disk for the next steps (`run_eval` on dev, then on report). It stops at the first step that fails.
4. After each step that changed `results/`, it uploads the whole `results/` folder to the same private repo, so the
   next session (or the Mac) can read it. `STAGE = "env_check"` changes nothing there, so it uploads nothing.

Needs the `HF_TOKEN` secret, internet on and a GPU runtime (setup: `docs/runtime.md`). An `ANTHROPIC_API_KEY` secret
is optional: it is loaded only for an `anthropic:` judge, kept in memory and never printed. `EXTRA_PIP` installs a
toolkit before the first step (for example `aligntune==0.1.31` for the fine-tune).

Model weights (`*.safetensors`, `*.bin`, `*.pt`, `*.pth`, any `merged/` folder), the `work/` folder and `.env` are
never uploaded to the dataset repo. The fine-tune sends its merged weights with its own `--push-to` to a PRIVATE model
repo. The A2 models are not uploaded at all. Making any model public is a separate step that only the owner takes.

In [ ]:
STAGE = "env_check"  # scripts/<STAGE>.py, run once with STAGE_ARGS when STEPS is empty (env_check uploads nothing)
STAGE_ARGS = ""  # its command-line arguments, e.g. "--arm A0 --split dev --model CohereLabs/tiny-aya-global"
STEPS = []  # or (script, args) pairs run in order in this session, e.g. [("run_resta", "..."), ("run_eval", "...")]
WORKSPACE_REPO = "YOUR-HF-USERNAME/msd-workspace"  # private HF dataset repo (docs/runtime.md)
EXTRA_PIP = ""  # pip arguments installed before the first step, e.g. "aligntune==0.1.31" (fine-tune only)
A1_REPO = ""  # the PRIVATE model repo that finetune.py --push-to wrote A1 to; set it to download A1 into the folder A1/

In [ ]:
import os

def find_token(name="HF_TOKEN"):  # the environment, then a Kaggle or Colab secret of that name; never printed
    if os.environ.get(name):
        return os.environ[name]
    try:  # Kaggle: Add-ons > Secrets, label = the name
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:  # Colab: Secrets (key icon), name = the name, Notebook access on
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

token = find_token()
assert token, "HF_TOKEN not found: add it as a Kaggle or Colab secret (see docs/runtime.md)"
os.environ["HF_TOKEN"] = token  # scripts and msd read it from here; it is never printed
print("HF_TOKEN found")
judge_key = find_token("ANTHROPIC_API_KEY")  # optional: only an anthropic: judge needs it; kept in memory only
if judge_key:
    os.environ["ANTHROPIC_API_KEY"] = judge_key  # for the scripts this notebook starts; never printed
print("ANTHROPIC_API_KEY found" if judge_key else "ANTHROPIC_API_KEY not set (only an anthropic: judge needs it)")

In [ ]:
import subprocess
import sys

from huggingface_hub import HfApi, hf_hub_download, snapshot_download

# The workspace repo holds model completions, harmful ones included: it must be private before anything is pulled.
if HfApi().repo_info(WORKSPACE_REPO, repo_type="dataset", token=token).private is not True:
    raise RuntimeError(f"{WORKSPACE_REPO} is not a private dataset repo, so nothing was pulled or uploaded (the results "
                       "hold model completions, including harmful ones). Make the repo private, then run again.")
# Keep exactly one wheel in wheels/ of the workspace repo; the last one by name is installed.
files = HfApi().list_repo_files(WORKSPACE_REPO, repo_type="dataset", token=token)
wheel = sorted(f for f in files if f.startswith("wheels/") and f.endswith(".whl"))[-1]
path = hf_hub_download(WORKSPACE_REPO, wheel, repo_type="dataset", token=token)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", path], check=True)
print("installed", wheel)
# The code, the configs, the data and every earlier result, so a step can read what another session wrote.
snapshot_download(WORKSPACE_REPO, repo_type="dataset", local_dir=".", token=token,
                  allow_patterns=["scripts/*", "pipelines/*", "configs/*", "data/**", "results/**"])
print("pulled scripts, pipelines, configs, data and results from", WORKSPACE_REPO)

In [ ]:
from huggingface_hub import snapshot_download

# run_resta.py --drifted needs A1 as a local folder; finetune.py --push-to put A1 in a PRIVATE model repo.
if A1_REPO.strip():
    snapshot_download(A1_REPO.strip(), local_dir="A1", token=token)  # the merged weights, about 6.7 GB
    print("downloaded", A1_REPO.strip(), "into the folder A1")
else:
    print("A1_REPO is empty: A1 is not downloaded (the ReSta steps need it as a folder)")

In [ ]:
from pathlib import Path

def results_state():  # every file under results/ with its size and time: shows whether a step wrote anything
    return {p.as_posix(): (p.stat().st_size, p.stat().st_mtime_ns) for p in Path("results").rglob("*") if p.is_file()}

def upload_results(before):  # the whole results/ folder, to the private repo only
    if results_state() == before:
        print("nothing to upload: results/ did not change")
        return
    # The gens files hold full model completions, harmful ones included: only ever send them to a private repo.
    private = HfApi().repo_info(WORKSPACE_REPO, repo_type="dataset", token=token).private
    if private is not True:
        raise RuntimeError(f"{WORKSPACE_REPO} is not a private dataset repo, so nothing was uploaded (the results "
                           "hold model completions, including harmful ones). Make the repo private, then run again.")
    # Never weights (the fine-tune uses its own --push-to to a private model repo), work/, crash leftovers or .env.
    never = ["*.safetensors", "*.bin", "*.pt", "*.pth", "work/**", "**/merged/**", "merged/**", "**/.*.partial/**",
             ".*.partial/**", ".env", "**/.env"]
    HfApi().upload_folder(folder_path="results", path_in_repo="results", ignore_patterns=never,
                          repo_id=WORKSPACE_REPO, repo_type="dataset", token=token)
    print("uploaded results/ to", WORKSPACE_REPO)

In [ ]:
import shlex
import subprocess
import sys

steps = list(STEPS) if STEPS else [(STAGE, STAGE_ARGS)]
for step in steps:  # every step is checked before the first one runs
    if not (isinstance(step, (tuple, list)) and len(step) == 2 and isinstance(step[0], str) and step[0].isidentifier()
            and isinstance(step[1], str)):
        raise ValueError(f"each step must be a (script name, arguments) pair of strings, such as "
                         f"(\"run_eval\", \"--arm A0 --split dev\"); got {step!r}")
if EXTRA_PIP.strip():  # a toolkit these steps need; a failed install stops the notebook here
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *shlex.split(EXTRA_PIP)], check=True)
for number, (script, args) in enumerate(steps, start=1):
    cmd = [sys.executable, f"scripts/{script}.py", *shlex.split(args)]
    print(f"step {number} of {len(steps)}, running:", " ".join(cmd))
    before = results_state()
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"step {number} ({script}) stopped with exit code {exc.returncode}; its message is above. The "
                           "steps after it did not run and nothing it wrote was uploaded. To keep what it wrote (for "
                           "example completions to resume from), run upload_results({}) in a new cell.") from None
    upload_results(before)
print("done:", len(steps), "step(s)")